<hr/>

# [StemCx Artificial Intelligence Workshop](https://donnievin.github.io/misc.html)
By: **Donovan Vincent Jr** - dvincen9@jh.edu <br/>
Estimated Workthrough Time: **3 hours**

<hr/>

<h1><font color="blue">Introduction to AI Workshop</font></h1>

This workbook is meant to be a high school friendly introduction to topics that one will encounter when venturing into the AI space. Many of these tools are extremely flexible, meaning they can be applied to nearly *any* problem you put your mind to! Feel free to work deeper with these techniques and other ones outlined throughout the workbook.

Topics: 
* General Classes of AI
* Linear Regression (Regression)
* Neural Networks
* Logistic Regression (Classification)
* Language Models
* Future of AI (Agents)

In [ ]:
!pip install transformers

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import plotly.express as px
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_moons

from transformers import GPT2Tokenizer, GPT2Model # If you want to train one from scratch
from transformers import GPT2Tokenizer, GPT2LMHeadModel # In the workshop so we can just generate for fun

np.random.seed(31100) # For reproducibility
torch.manual_seed(31100) # For reproducibility

# <font color='lightblue'> General Classes of AI</font>

 |                | Supervised     |         Unsupervised     |
 |:---------------|:--------------:|:------------------------:|
 | **Continuous** | Regression     | Dimensionality Reduction |
 | **Discrete**   | Classification | Clustering               |   

# <font color='lightblue'> Linear Regression</font>

> Goal (Core Idea)
 
 Given some input, x, we can <font color='green'>fit a line</font>, that will be able to predict an observation, y.

<div align="center">

$$
y = mx + b
$$

</div>


But what if we want the *same input* to predict <font color='green'>multiple outputs</font>. We would need 2 equations:

<div align="center">

$$
y_1 = m_1x + b_1
$$

$$
y_2 = m_2x + b_2
$$

</div>


Example: Given that x=zip code, I want to predict $y_1$ = price and $y_2= n_{beds}$

---

> Linear ALgebra Approach


We can write all of these equations in a compact way using <font color='green'>Linear Algebra Notation</font>. Subscripts are written *row* then *column*. In general, this will be: 

$$
{
%\defined
\left(\begin{array}{c}
y_1 \\
y_2 \\
\vdots \\
y_n \\
\end{array}
\right)}
=

{
\left(\begin{array}{cccc}
x_{1,p-1} & ... & x_{1,1} & 1 \\
x_{2,p-1} & ... & x_{2,1} & 1 \\
\vdots \\
x_{n,p-1} & ... & x_{n,1} & 1 \\
\end{array}
\right)}


{
\left(\begin{array}{c}
b_1 \\
b_2 \\
\vdots \\
b_p \\
\end{array}
\right)}
$$


For the previous example (n=2, p=2), this would become: 

$$
{
%\defined
\left(\begin{array}{c}
y_1 \\
y_2 \\
\end{array}
\right)}
=

{
\left(\begin{array}{cccc}
x_{1,1} & 1 \\
x_{2,1} & 1 \\
\end{array}
\right)}


{
\left(\begin{array}{c}
b_1 \\
b_2 \\
\end{array}
\right)}
$$


<font color='purple'>Q: What is n, and p in this case:</font>


--- 

> Solving multiple equations simultaneously

Written compactly:

$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} $$

This one expression represents **every equation at once**. Now, instead of solving each line separately, we can solve for all coefficients $\boldsymbol{\beta}$ in a single operation using the **Normal Equation**:

$$\boxed{\boldsymbol{\hat{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}}$$

This finds the $\boldsymbol{\beta}$ that minimizes the total squared error across **all observations simultaneously** — no matter how many inputs or outputs we have.

In [ ]:
def linear_regression(X,y):
    # X must be nxm, y must be nx1
    return np.linalg.inv(X.T @ X) @ (X.T @ y)


def linear_regression_accuracy(prediction, true_values):
    ss_res = np.sum((true_values - prediction) ** 2)
    ss_tot = np.sum((true_values - np.mean(true_values)) ** 2)
    r2 = np.abs((1 - ss_res / ss_tot))*100
    return r2

## Ex 1: Height vs Weight

In [ ]:
###################### YOU PICK THESE ######################
noise = 50
train_test_split = 0.8
###################### YOU PICK THESE ######################

In [ ]:
np.random.seed(31100) # For reproducibility

# Synthetic Data (W = 3H + 5)
height = np.arange(1,72) #in
weight = 3*height + 5 + np.random.normal(0,noise, size=len(height)) # some noise to make it harder

# Clean it up to make it easier to use
Example1_data = pd.DataFrame({
    "intercept": np.ones(len(height)),
    "height":height,
    "weight":weight,
})

# Graph the data
plt.scatter(data=Example1_data, x="height", y="weight")
plt.axvline(x=60, color='red', linestyle='--', label="5ft", alpha=0.2)
plt.xlabel("Height (in)", fontsize=20)
plt.ylabel("Weight (lbs)", fontsize=20)
plt.title("Fake data for LR", fontsize=20)
plt.show()

In [ ]:
# Train / Test split
train_data = Example1_data.sample(frac=train_test_split,random_state=31100)
test_data = Example1_data.drop(train_data.index)

# Training 
X = train_data[["intercept", "height"]].values
y = train_data[["weight"]].values
B = linear_regression(X,y)

# Prediction (y = XB)
y_pred = test_data[["intercept", "height"]].values @ B
y_test = test_data[["weight"]].values
accuracy = linear_regression_accuracy(y_pred, y_test)

# Plotting the fitted line
plt.scatter(x=y_pred, y=y_test)
plt.plot([0,220], [0,220], color='grey', alpha=0.3)
plt.xlabel("True Values", fontsize=20)
plt.ylabel("Predicted Values", fontsize=20)
plt.title(f"Accuracy = {accuracy:.2f}%", fontsize=20)
plt.show()


In [ ]:
# Predict a new members weight
new_height = 63 #inches 
new_weight = 200 #lbs


# Plot it on the graph
plt.scatter(data=Example1_data, x="height", y="weight")
plt.scatter(x=new_height, y=new_weight, color='red', label='New person')

# Create a sorted set of heights for the regression line
x_line = np.sort(train_data["height"].values)
X_line = np.column_stack((np.ones(len(x_line)), x_line))
y_line = X_line @ B

# Plot regression line
plt.plot(x_line, y_line, color='orange', linewidth=2, label='Regression line')
plt.xlabel("Height (in)", fontsize=20)
plt.ylabel("Weight (lbs)", fontsize=20)
plt.title("Fake data for LR", fontsize=20)
plt.legend()
plt.show()


<font color='purple'>Q1a: How does the noise level affect the accuracy?</font>
---


<font color='purple'>Q1b: How does the train test split affect the accuracy?</font>
---

## Ex 2: Housing Prices

In [ ]:
housing_url = 'https://alanagresti.com/glm/data/Houses.dat'
housing_df = pd.read_csv(housing_url, sep='\s+', usecols=['beds', 'baths', 'price']) # Index(['case', 'taxes', 'beds', 'baths', 'new', 'price', 'size'], dtype='object')
housing_df['intercept'] = np.ones(len(housing_df['beds']))

interactive_fig = go.Figure(
    data=[
        go.Scatter3d(
            x=housing_df['beds'],
            y=housing_df['baths'],
            z=housing_df['price'],
            mode='markers',
            marker=dict(
                size=4,
                opacity=0.7
            )
        )
    ]
)

interactive_fig.update_layout(
    width=700,
    height=400,
    margin=dict(l=0, r=0, b=0, t=30),
    scene=dict(
        xaxis_title='Beds',
        yaxis_title='Baths',
        zaxis_title='Price',
        aspectmode='cube',   # keeps plot centered
    )
)

interactive_fig.show()

In [ ]:
# Graph beds vs price and baths vs price
housing_fig, housing_axes = plt.subplots(1,2, figsize=(12,4))

housing_axes[0].scatter(data=housing_df, x='beds', y='price')
housing_axes[0].set_ylabel("Price in Thousands")
housing_axes[0].set_xlabel("Number of Bed Rooms")
housing_axes[0].set_xticks(np.arange(1,6,1))

housing_axes[1].scatter(data=housing_df, x='baths', y='price')
housing_axes[1].set_xlabel("Number of Bath Rooms")
housing_axes[1].set_xticks(np.arange(0,6,1))

plt.tight_layout()
plt.show()

In [ ]:
house_train_data = housing_df.sample(frac=0.8,random_state=31100)
house_test_data = housing_df.drop(train_data.index)

# Training 
X_house = house_train_data[["intercept", "beds", "baths"]].values
y_house = house_train_data[["price"]].values
B_house = linear_regression(X_house,y_house)

# Prediction (y = XB)
y_pred_house_price = house_test_data[["intercept", "beds", "baths"]].values @ B_house
y_test_house_price = house_test_data[["price"]].values
accuracy_house_price = linear_regression_accuracy(y_pred_house_price, y_test_house_price)

# Plotting the fitted line
plt.scatter(x=y_pred_house_price, y=y_test_house_price)
plt.plot([0,500], [0,500], color='grey', alpha=0.3)
plt.xlabel("True Values", fontsize=20)
plt.ylabel("Predicted Values", fontsize=20)
plt.title(f"Accuracy = {accuracy_house_price:.2f}%", fontsize=20)
plt.show()


<font color='purple'>Q2a: How do you think we can improve our accuracy on the housing dataset?</font>
---

# <font color='lightblue'> Neural Networks </font>

> Goal

Linear regression is great, but it can only get us so far. What if the data IS NOT LINEAR.
NN allow us to predict a wider range of functions using combinations of linear and non-linear pieces


<img src='https://www.ibm.com/adobe/dynamicmedia/deliver/dm-aid--3ab84ac9-60c1-49b2-ba0d-f6260444aa10/iclh-diagram-batch-01-03-deepneuralnetwork.png?preferwebp=true&width=1584' width=800 align=center>

---

> Activation Functions

These are the non-linear piece of the neural network

<img src='https://miro.medium.com/v2/resize:fit:1400/format:webp/1*6pa6uqH4B0Ia4FQ3WzsV_A.png' width=800 align=center>


--- 


> Learning 


Machine learning models <font color='green'>learn</font> when they make a mistake and we <font color='green'>give feedback on how bad</font> was their mistake. There are many algorithms, but the most common is [Gradient Descent 3B1B](https://www.youtube.com/watch?v=IHZwWFHWa-w)


$$
w_{new} = w_{old} - \eta \nabla L
$$

* where: 
    * $\eta $ = learning rate
    * $\nabla L$ = how wrong was the model

--- 

> Advantages and Disadvantages


 |       | Advantages                   |  Disadvantages                            |
 |:------|:----------------------------:|:-----------------------------------------:|
 | **1** | Finds hidden patterns        | Intrepretability (Black box process)      |
 | **2** | Works with Non-linear data   | More parameters to pick (depth, width, lr, activation function, etc.)|
 | **3** | Better for Large datasets    | -                                         |

 

--- 

> PyTorch

This lets us build Neural Networks in a fun way!


## Ex 3: Revisiting House Data

In [ ]:
###################### YOU PICK THESE ######################
h_dim = 4
n_layers = 8
activation_function = nn.ReLU() # Also try: nn.Sigmoid(), nn.Tanh(), nn.SiLU()
###################### YOU PICK THESE ######################

In [ ]:
# Convert data to tensors (train / test split)
X_house_train_tensor = torch.from_numpy(house_train_data[["intercept", "beds", "baths"]].values).float() # 80x3
y_house_train_tensor = torch.from_numpy(house_train_data[["price"]].values).float().reshape(-1, 1) #80x1


X_house_test_tensor = torch.from_numpy(house_test_data[["intercept", "beds", "baths"]].values).float() #80x3
y_house_test_tensor = torch.from_numpy(house_test_data[["price"]].values).float().reshape(-1, 1) #80x1


# Build a Neural Network
layers = []
input_size = X_house_train_tensor.shape[1]

for i in range(n_layers):
    layers.append(nn.Linear(input_size, h_dim)) # Y = XB
    layers.append(activation_function) # Non-linear portion
    input_size = h_dim 

layers.append(nn.Linear(h_dim, 1)) # Final Y=XB
house_neural_network = nn.Sequential(*layers)

print(house_neural_network)



In [ ]:
# Loss function + optimizer
loss_fn = nn.MSELoss()
optimizer = optim.Adam(house_neural_network.parameters(), lr=0.01)

# Training loop
epochs = 1000

for epoch in range(epochs):
    predictions = house_neural_network(X_house_train_tensor) # # Forward pass

    loss = loss_fn(predictions, y_house_train_tensor)
    optimizer.zero_grad() # Clear old gradients
    loss.backward() # Backpropagation
    optimizer.step() # Update weights

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.2f}")

In [ ]:
house_neural_network.eval()

with torch.no_grad(): 
    y_pred_nn = house_neural_network(X_house_test_tensor)

accuracy_house_price_nn = linear_regression_accuracy(y_pred_nn.detach().numpy(), y_house_test_tensor.detach().numpy())


# Plotting the fitted line
plt.scatter(x=y_pred_nn, y=y_house_test_tensor)
plt.plot([0,500], [0,500], color='grey', alpha=0.3)
plt.xlabel("True Values", fontsize=20)
plt.ylabel("Predicted Values", fontsize=20)
plt.title(f"Accuracy = {accuracy_house_price_nn:.2f}%", fontsize=20)
plt.show()

<font color='purple'>Q3a: How does the hidden dimension size change accuracy? </font>
---

<font color='purple'>Q3b: How does the number of layers change accuracy?</font>
---

<font color='purple'>Q3c: Which activation function gets the highest accuracy?</font>
---

# <font color='lightblue'> Logistic Regression</font>

> Log-odds

We can easily to quantify the <font color='green'>ratio of belonging in a desired class versus in a different class</font> and we call this the odds ratio. We take a logarithm for "numerical stability" in the computer, meaning the numbers will not get too small that the compueter can no longer calculate it.

<div align="center">

$$
log(\frac{P}{1-P})
$$

</div>

--- 

> Extracting probabilities

Using the linear regression formula that we saw above, we can now predict this log-odds ratio using the same technique from above. However, instead of getting one number, we can pull out the <font color='green'>probability of belong to a class</font>.


$$
log(\frac{P}{1-P}) = X\beta
$$


Solving for P, we get: 

$$
P = \frac{1}{1+e^{-X\beta}}
$$

This is known as the <font color='green'>logistic function</font>.

---

> Softmax


If we have multiple categories, this method can still be used to get <font color='green'> Probabilities of belong to each class</font>.

$$
P_c = \frac{e^{z_i}}{1+\sum_{i=1}^C e^{z_i}}
$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Ex 4: The sigmoid curve

In [ ]:
# Generate z values (linear combination input range)
z = np.linspace(-10, 10, 200)

# Apply sigmoid
sigma = sigmoid(z)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(z, sigma, color='red', linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='σ = 0.5')
plt.axvline(0, color='gray', linestyle=':', linewidth=1)
plt.xlabel("z (linear combination)", fontsize=20)
plt.ylabel("σ(z)", fontsize=20)
plt.title("Sigmoid Function", fontsize=20)
plt.legend(fontsize=14)
plt.tight_layout()
plt.show()

## Ex 5: Simple logistic regression

In [ ]:
# Data online
hours_studied = [0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 1.75, 2.00, 2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 4.00, 4.25, 4.50, 4.75, 5.00, 5.50]
passed_or_failed = [0,0,0,0,0,0,1,0,1,0,1,0,1,0,1,1,1,1,1,1]

# Clean it up to make it easier to use
Example5_data = pd.DataFrame({
    "intercept": np.ones(len(hours_studied)),
    "hours_studied":hours_studied,
    "passed_or_failed":passed_or_failed,
})

# Graph the data
plt.scatter(data=Example5_data, x="hours_studied", y="passed_or_failed")
plt.xlabel("Hours spent studying", fontsize=20)
plt.ylabel("Passed or Failed", fontsize=20)
plt.title("Logistic Regression Data", fontsize=20)
plt.show()


In [ ]:
# Fit logistic regression
X = Example5_data[["hours_studied"]].values
y = Example5_data["passed_or_failed"].values

model = LogisticRegression()
model.fit(X, y)

# Prediction grid
hours = np.linspace(0, 6, 100).reshape(-1, 1)
P_LR = model.predict_proba(hours)[:, 1]

# Plot
plt.scatter(hours_studied, passed_or_failed)
plt.plot(hours, P_LR, color="red", linewidth=2)
plt.xlabel("Hours spent studying", fontsize=20)
plt.ylabel("Passed or Failed", fontsize=20)
plt.title("Logistic Regression Data", fontsize=20)
plt.show()

## Ex 6: Breast Cancer (Scikit learn)

In [ ]:
from sklearn.datasets import load_breast_cancer

# Load data
X, y = load_breast_cancer(return_X_y=True)

# Use only one feature so we can plot the curve
X = X[:, [0]]   # mean radius

# Fit logistic regression
model = LogisticRegression(max_iter=10000)
model.fit(X, y)

# Generate smooth x values
x_curve = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)

# Predict probabilities
y_curve = model.predict_proba(x_curve)[:, 1]

# Plot
plt.scatter(X, y, alpha=0.4, label="Data")
plt.plot(x_curve, y_curve, color="red", linewidth=3,
         label="Logistic Regression")

plt.xlabel("Mean Radius")
plt.ylabel("Probability of Positive Breast Cancer")
plt.legend()
plt.show()

In [ ]:
print(model.intercept_)
print(model.coef_)

<font color='purple'>Q 4a: Why is the log odds better than just using a probability directly?</font>
---

# <font color='lightblue'> Language Models </font>

> Goal 

Language models are just a fancy combination of linear and logistic regression. 

We have a wide class of words, and we want to predict the probability that


---

> Tokenization 


We can't do math on words, so we need a <font color='green'>look up table</font> to get from words to numbers and back. For example

* Key:
    * black=0
    * cat=1
    * hat=17
    * in=23
    * is=27
    * the=35

* "the cat in the hat" becomes: [35, 1, 23, 35, 17]. 


<font color='purple'>Q: What would [35, 0, 1] translate to?</font>



--- 

> Generation

Using the tools we saw before (linear and logistic regression), we can now see how generation is possible with a language model 

* *Step 1*: <font color='green'>Tokenization</font> $\implies$ Turn words to numbers to do math
* *Step 2*: <font color='green'>Embeddings</font> $\implies$ Creates rich representations
* *Step 3*: <font color='green'>Linear Regression</font> $\implies y=X \beta$
* *Step 4*: <font color='green'>Logistic Regression</font>: $\implies$ Words are categories, which category has the highest probability
* *Step 5*: <font color='green'>Repeat</font>: $\implies$ Keep doing this until you have a complete sentence

As a math equation, this can be represented as: 

$$
P(x_i | x_{<i})
$$

Read as "The probability at position i given all context less than i"


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

text = "Once upon a time"
inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=100,
    do_sample=True,
    temperature=0.8
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

# <font color='lightblue'> Future of AI</font>

The future is agents 
---


<img src='https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F3613f360926fae004521197488623465eb0cd751-1920x1035.png&w=3840&q=75' width=800 align=center>

# Further References

1. [Kaggle](https://www.kaggle.com): Tons of other datasets and practice algorithms
2. [Huggingface](https://huggingface.co): Tons of datasets and practice algorithms
3. [Sci-kit learn](): Better, more efficient implementations of these common algorithms
    * [Linear Regression]():
    * [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html):
4. [Alan Agresti](): More statistics stuff, but also has tons of datasets
5. [Andrej Karpathy's GPT from scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY): The transformer guy
6. [Gradient Descent 3B1B](https://www.youtube.com/watch?v=IHZwWFHWa-w): Beautiful intuitive videos
7. [Claude Code](https://claude.com/product/claude-code): One of the leading agentic platforms right now
8. [Cursor](): One of the leading agentic platforms right now
9. [Codex](): One of the leading agentic platforms right now